In [2]:
# ============================================================
# AUTOSUPPLY-AI - SUPPLY PLANNING
# CELL 1: PROJECT SETUP
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# PROJECT ROOT
# ------------------------------------------------------------

PROJECT_ROOT = Path(r"F:\AutoSupply-AI")

# ------------------------------------------------------------
# NOTEBOOKS FOLDER
# All generated CSV files are stored here
# ------------------------------------------------------------

NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"

# ------------------------------------------------------------
# INPUT FILE
# Exact filename visible in the project Explorer
# ------------------------------------------------------------

INVENTORY_FILE = NOTEBOOKS_DIR / "inventory_optimization.csv"

# ------------------------------------------------------------
# OUTPUT FILE
# ------------------------------------------------------------

OUTPUT_FILE = NOTEBOOKS_DIR / "autosupply_supply_plan.csv"

# ------------------------------------------------------------
# DISPLAY PATHS
# ------------------------------------------------------------

print("=" * 70)
print("AUTOSUPPLY-AI - SUPPLY PLANNING")
print("=" * 70)

print("\nProject Root:")
print(PROJECT_ROOT)

print("\nInput File:")
print(INVENTORY_FILE)

print("\nOutput File:")
print(OUTPUT_FILE)

# ------------------------------------------------------------
# VERIFY PATHS
# ------------------------------------------------------------

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"Project root not found:\n{PROJECT_ROOT}"
    )

if not NOTEBOOKS_DIR.exists():
    raise FileNotFoundError(
        f"Notebooks folder not found:\n{NOTEBOOKS_DIR}"
    )

if not INVENTORY_FILE.exists():
    raise FileNotFoundError(
        f"Inventory optimization file not found:\n{INVENTORY_FILE}"
    )

print("\n" + "=" * 70)
print("✓ PROJECT ROOT FOUND")
print("✓ NOTEBOOKS FOLDER FOUND")
print("✓ inventory_optimization.csv FOUND")
print("=" * 70)

AUTOSUPPLY-AI - SUPPLY PLANNING

Project Root:
F:\AutoSupply-AI

Input File:
F:\AutoSupply-AI\notebooks\inventory_optimization.csv

Output File:
F:\AutoSupply-AI\notebooks\autosupply_supply_plan.csv

✓ PROJECT ROOT FOUND
✓ NOTEBOOKS FOLDER FOUND
✓ inventory_optimization.csv FOUND


In [3]:
# ============================================================
# CELL 2: LOAD AND VALIDATE INVENTORY DATA
# ============================================================

inventory_df = pd.read_csv(INVENTORY_FILE)

print("=" * 70)
print("INVENTORY DATA LOADED")
print("=" * 70)

print("\nShape:")
print(inventory_df.shape)

print("\nColumns:")
print(list(inventory_df.columns))

print("\nFirst 5 rows:")
display(inventory_df.head())

print("\nMissing Values:")
print(inventory_df.isnull().sum())

print("\nData Types:")
print(inventory_df.dtypes)

INVENTORY DATA LOADED

Shape:
(203, 10)

Columns:
['state', 'vehicle_group', 'average_monthly_demand', 'demand_std', 'total_forecast_demand', 'forecast_months', 'safety_stock', 'reorder_point', 'recommended_inventory', 'inventory_priority']

First 5 rows:


,state,vehicle_group,average_monthly_demand,demand_std,total_forecast_demand,forecast_months,safety_stock,reorder_point,recommended_inventory,inventory_priority
0,Rajasthan,Two Wheeler,1577691.25,146784.64,9466147.50,6,242194.66,1819885.91,1819885.91,HIGH
1,Rajasthan,Passenger Vehicle,420453.71,10295.42,2522722.25,6,16987.44,437441.15,437441.15,HIGH
2,Uttar Pradesh,Two Wheeler,289612.42,10015.11,1737674.53,6,16524.93,306137.36,306137.36,HIGH
3,Maharashtra,Two Wheeler,148345.68,11630.96,890074.08,6,19191.09,167536.77,167536.77,HIGH
4,Tamil Nadu,Two Wheeler,143606.91,11661.99,861641.46,6,19242.28,162849.19,162849.19,HIGH



Missing Values:
state                     0
vehicle_group             0
average_monthly_demand    0
demand_std                0
total_forecast_demand     0
forecast_months           0
safety_stock              0
reorder_point             0
recommended_inventory     0
inventory_priority        0
dtype: int64

Data Types:
state                      object
vehicle_group              object
average_monthly_demand    float64
demand_std                float64
total_forecast_demand     float64
forecast_months             int64
safety_stock              float64
reorder_point             float64
recommended_inventory     float64
inventory_priority         object
dtype: object


In [4]:
# ============================================================
# CELL 3: CALCULATE SUPPLY REQUIREMENTS
# ============================================================

# Create a working copy
supply_df = inventory_df.copy()

# ------------------------------------------------------------
# 1. Calculate current required inventory
# ------------------------------------------------------------

supply_df["required_inventory"] = (
    supply_df["recommended_inventory"]
)

# ------------------------------------------------------------
# 2. Calculate supply requirement
# ------------------------------------------------------------
# Supply requirement represents the inventory level that
# should be available to satisfy forecasted demand and
# maintain the calculated safety buffer.

supply_df["supply_requirement"] = (
    supply_df["total_forecast_demand"]
    + supply_df["safety_stock"]
)

# ------------------------------------------------------------
# 3. Calculate inventory coverage ratio
# ------------------------------------------------------------

supply_df["inventory_coverage_ratio"] = (
    supply_df["recommended_inventory"]
    / supply_df["average_monthly_demand"]
)

# ------------------------------------------------------------
# 4. Calculate monthly replenishment requirement
# ------------------------------------------------------------

supply_df["monthly_replenishment_requirement"] = (
    supply_df["supply_requirement"]
    / supply_df["forecast_months"]
)

# ------------------------------------------------------------
# 5. Round numerical values
# ------------------------------------------------------------

numeric_columns = [
    "required_inventory",
    "supply_requirement",
    "inventory_coverage_ratio",
    "monthly_replenishment_requirement"
]

for column in numeric_columns:
    supply_df[column] = supply_df[column].round(2)

# ------------------------------------------------------------
# DISPLAY RESULTS
# ------------------------------------------------------------

print("=" * 70)
print("SUPPLY REQUIREMENT CALCULATION")
print("=" * 70)

print("\nNumber of State-Vehicle Combinations:")
print(len(supply_df))

print("\nTotal Supply Requirement:")
print(f"{supply_df['supply_requirement'].sum():,.2f}")

print("\nAverage Monthly Replenishment Requirement:")
print(
    f"{supply_df['monthly_replenishment_requirement'].mean():,.2f}"
)

print("\nTop 10 Supply Requirements:")

display(
    supply_df[
        [
            "state",
            "vehicle_group",
            "total_forecast_demand",
            "safety_stock",
            "supply_requirement",
            "monthly_replenishment_requirement"
        ]
    ]
    .sort_values(
        "supply_requirement",
        ascending=False
    )
    .head(10)
)

print("\n" + "=" * 70)
print("✓ SUPPLY REQUIREMENTS CALCULATED")
print("=" * 70)

SUPPLY REQUIREMENT CALCULATION

Number of State-Vehicle Combinations:
203

Total Supply Requirement:
27,293,778.70

Average Monthly Replenishment Requirement:
22,408.69

Top 10 Supply Requirements:


,state,vehicle_group,total_forecast_demand,safety_stock,supply_requirement,monthly_replenishment_requirement
0,Rajasthan,Two Wheeler,9466147.50,242194.66,9708342.16,1618057.03
1,Rajasthan,Passenger Vehicle,2522722.25,16987.44,2539709.69,423284.95
2,Uttar Pradesh,Two Wheeler,1737674.53,16524.93,1754199.46,292366.58
3,Maharashtra,Two Wheeler,890074.08,19191.09,909265.17,151544.19
4,Tamil Nadu,Two Wheeler,861641.46,19242.28,880883.74,146813.96
5,Rajasthan,Three Wheeler,768975.80,17116.60,786092.40,131015.40
6,Karnataka,Two Wheeler,696028.61,14806.87,710835.48,118472.58
7,Bihar,Two Wheeler,623129.37,22696.43,645825.80,107637.63
8,Rajasthan,Goods Vehicle,581692.04,14039.61,595731.65,99288.61
9,Madhya Pradesh,Two Wheeler,564443.55,19306.01,583749.56,97291.59



✓ SUPPLY REQUIREMENTS CALCULATED
